# **Otimização Linear: Trabalho Final**


**SME0211**

##### **Membros**
Francisco Maian:  14570890 \\
Gustavo Moreira:  5244057 \\
Julia Pravato:  14615054 \\
Murilo Lirani:  11234673 \\


##### **Relatório**

https://www.overleaf.com/read/bdytngppmzxq#e66176

### **Bibliotecas**

In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.optimize import linprog
np.random.seed(10)

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### **Funções Geradores de Problema**

In [ ]:
# Calculo dos parâmetros da função beta (baseando na média desvio padrão)
def calc_beta_params(mean, std):
    var = std**2
    alpha = mean * ((mean * (1 - mean)) / var - 1)
    beta = (1 - mean) * ((mean * (1 - mean)) / var - 1)
    return (alpha, beta)

In [ ]:
def gerar_tensor_aleatorio(size = (3, 5), B_params=(0.5,0.1), sparsity=0):
    alpha, beta = calc_beta_params(*B_params)

    # Gera uma matriz de valores aleatórios uniformes entre 0 e 1, com tamanho especificado por 'size'
    relevancia = np.random.rand(*size)

    # Define posições para zerar com base na sparsidade definida, e zera os valores nas posições
    zeros_pos = np.random.rand(*size) < sparsity
    relevancia[zeros_pos] = 0

    # Cria uma lista de vetores de limites (default: todos 1)
    limites = [np.ones(dim) for dim in size]

    # Ajuste do segundo limite com valores de uma distribuição Beta
    if len(size) > 1:
        limites[1] = np.random.beta(alpha, beta, size[1])
    return relevancia, limites

In [ ]:
def traducao_variaveis(relevancia, limites):
    # Obter o número de dimensões e os tamanhos de cada dimensão
    shape = relevancia.shape
    num_dims = len(shape)
    total_elements = np.prod(shape)

    # Raveliza o tensor em um vetor
    r = relevancia.ravel()

    # Inicializa uma lista para armazenar as matrizes S de cada dimensão
    S_list = []

    for dim, dim_size in enumerate(shape):
        # Cria uma matriz S para a dimensão atual
        repeat_factors = np.prod(shape[:dim], dtype=int)
        tile_factors = np.prod(shape[dim + 1:], dtype=int)

        row_indices = np.repeat(np.arange(dim_size), tile_factors)
        row_indices = np.tile(row_indices, repeat_factors)
        col_indices = np.arange(total_elements)

        S = np.zeros((dim_size, total_elements))
        S[row_indices, col_indices] = 1

        # Adiciona a matriz S à lista
        S_list.append(S)

    # Empilha todas as matrizes S verticalmente
    A = np.vstack(S_list)

    # Cria o array b a partir dos limites
    b = np.hstack([limite.ravel() for limite in limites])

    return A, b, r

### **Forma Padrão**

In [ ]:
def forma_padrao(A_, b_, r_):
    # Dimensões da matriz A_
    m, n = A_.shape

    # Atualiza o vetor e custos, removendo as colunas associadas a custos nulos no vetor de custos r_
    mask = r_ != 0
    A_ = A_[:, mask]
    r_ = r_[mask]

   # Adiciona variáveis de folga
    A = np.hstack([A_, np.eye(m)])

    # Constrói o vetor de custos com os custos negativos e zeros para as variáveis de folga
    c = - np.concatenate([r_, np.zeros(m)])
    b = b_
    return A, b, c

### **Gerador de Problema**

In [ ]:
# Exemplo de uso
size = [6, 15]  # Dimensão do tensor
relevancia, limites = gerar_tensor_aleatorio(size, (0.8, 0.1), 0.4)

A_, b_, r_ = traducao_variaveis(relevancia, limites)

In [ ]:
tabela_relevancia = pd.DataFrame(relevancia, index=limites[0], columns=limites[1])
tabela_relevancia

Com isso, foi gerado um problema

\begin{aligned}
\underline{P}: \quad & \max & \underline{r}^T \ x \\
                & \text{sujeito a} & \underline{A} x \leq b, \\
                &                        & x \geq 0.
\end{aligned}

##### **Elementos de $\underline{P}$**

In [ ]:
A_

In [ ]:
b_

In [ ]:
r_

In [ ]:
A_.shape

In [ ]:
r_.shape

##### **Aplicando o Problema na Forma Padrão**

In [ ]:
A, b, c = forma_padrao(A_, b_, r_)

Resultando em:
\begin{aligned}
\text{P:} \quad & \min & c^T \ x \\
                & \text{sujeito a} & A x = b \\
                &                        & x \geq 0
\end{aligned}

##### **Aplicando o Problema na Forma Padrão**

In [ ]:
A

In [ ]:
c

In [ ]:
b

### **SimpleX Revisado utilizando linprog do Scipy**

In [ ]:
res = linprog(c, A_eq=A, b_eq=b, bounds=(0, None), method='simplex')
print("\n",
      "Custo:", res.fun, "\n",
      "Interações:", res.nit, "\n"
      "Solução:", "\n", np.round(res.x, 5)
)

### **Programando o SimpleX com a Regra de Bland para evitar ciclagem**

In [ ]:
def simplex(c, A, b):

  #Construindo o tableau:
  tableau = A.copy()
  valor_variaveis = b.copy()
  custo_reduzido = c.copy()
  custo_final = 0
  interacoes = 0

  variaveis_base = []
  for i in range(c.size - b.size, c.size):
    variaveis_base.append(i)

  fim = False
  while(fim == False):
    for i in range(custo_reduzido.size):
      #Pega o primeiro custo negativo que encontrar
      if (custo_reduzido[i] < 0):
        #Achar melhor valor pra substituir, utilizando a regra de bland
        custos = []
        for index_base in range(valor_variaveis.size):
          if (tableau[index_base][i] > 0):
            custos.append(valor_variaveis[index_base] / tableau[index_base][i])
          else:
            #Aplicando o valor infinito nas linhas do tableau que Aj é menor ou igual a zero, para o programa nunca pegar esses valores na hora de trocar de variável base
            custos.append(float('inf'))

        #Pegando o menor index dentre as variáveis com o menor custo para evitar ciclagem (Regra de Bland)
        menor = min(custos)
        menores = []
        menor_index = custos.index(menor)
        for index_bland in range(custos.index(menor), len(custos)):
          if custos[index_bland] == menor:
            if (index_bland < menor_index):
              menor_index = index_bland

        #Realizando a troca do tableau, com o menor_index adquirido através da Regra de Bland
        #i == variavel pra entrar na base
        #menor_index == variavel pra sair da base

        #Trocando os valores da linha da variavel a sair da base
        valor_divisao = tableau[menor_index][i]
        for index_variaveis in range(custo_reduzido.size):
          tableau[menor_index][index_variaveis] /= valor_divisao
        valor_variaveis[menor_index] /= valor_divisao

        #Trocando os valores das outras variaveis pra definir a nova base
        for index_base in range(valor_variaveis.size):
          #Se for a variavel que vai sair da base não deixa 0 a coluna
          if (index_base != menor_index):
            valor_soma = -tableau[index_base][i]
            for index_variaveis in range(custo_reduzido.size):
              tableau[index_base][index_variaveis] += tableau[menor_index][index_variaveis] * valor_soma
            valor_variaveis[index_base] += valor_variaveis[menor_index] * valor_soma

        #Zerando o custo reduzido da nova base
        valor_soma = -custo_reduzido[i]
        for index_variaveis in range(custo_reduzido.size):
          custo_reduzido[index_variaveis] += tableau[menor_index][index_variaveis] * valor_soma
        custo_final += valor_variaveis[menor_index] * valor_soma

        #Trocando a base
        variaveis_base[menor_index] = i
        interacoes += 1
        break

      #Se passou do ultimo valor, quer dizer que não há mais custos reduzidos negativos, então o SimpleX chegou ao fim
      if(i == custo_reduzido.size - 1):
        fim = True

  solucao = np.zeros(c.size)
  for i in range(len(variaveis_base)):
    solucao[variaveis_base[i]] = valor_variaveis[i]

  return -custo_final, solucao, interacoes

In [ ]:
custo, solucao, interacoes = simplex(c, A, b)
print("\n",
      "Custo:", custo, "\n",
      "Interações:", interacoes, "\n"
      "Solução:", "\n", np.round(solucao, 5)
)

Pode-se ver que encontramos a mesma solução ótima para os dois problemas, mas suas iterações diferem


### **Experimentos**

##### **Teste Singular**

In [ ]:
def teste_singular(size, B_params, sparsity):
  # Tensor de relevância e limites aleatórios
    relevancia, limites = gerar_tensor_aleatorio(size, B_params, sparsity)

    A_, b_, r_ = traducao_variaveis(relevancia, limites)
    A, b, c = forma_padrao(A_, b_, r_)
    varsize = A.shape[1] - A.shape[0] #numero de variaveis consideradas

    start_time_scipy = time.time()
    res_scipy = linprog(c, A_eq=A, b_eq=b, bounds=(0, None), method='simplex')
    end_time_scipy = time.time()
    custo_scipy, nit_scipy = res_scipy.fun, res_scipy.nit
    tempo_scipy = end_time_scipy - start_time_scipy

    start_time_au = time.time()
    res_au = simplex(c, A, b)
    end_time_au = time.time()
    custo_au, nit_au = res_au[0], res_au[2]
    tempo_au = end_time_au - start_time_au

    return size, B_params, custo_scipy, nit_scipy, tempo_scipy, custo_au, nit_au, tempo_au, sparsity, varsize

In [ ]:
 size, B_params, custo_scipy, nit_scipy, tempo_scipy, custo_au, nit_au, tempo_au, sparsity, varsize = teste_singular((10, 30), (0.5, 0.1) , 0)
 print("\n",
       "Dimensão do tensor:", size, "\n",
       "B_params:", B_params, "\n",
       "Custo (Scipy):", custo_scipy, "\n",
       "Iterações (Scipy):", nit_scipy, "\n",
       "Tempo (Scipy):", tempo_scipy, "\n",
       "Custo (SimpleX com a Regra de Bland):", custo_au, "\n",
       "Iterações (SimpleX com a Regra de Bland):", nit_au, "\n",
       "Tempo (SimpleX com a Regra de Bland):", tempo_au, "\n",
       "Sparsity:", sparsity, "\n",
       "Tamanho Variáveis:", varsize
     )

##### **Amostragem**

In [ ]:
def amostragem(sample_size, size = (6, 10), B_params = (0.5, 0.1), sparsity = 0):
    results = []
    for _ in range(sample_size):
        result = teste_singular(size, B_params, sparsity)
        results.append(result)

    columns = [
        "size", "B_params","custo_scipy", "nit_scipy",
        "tempo_scipy", "custo_SimpleXBland", "nit_SimpleXBland", "tempo_SimpleXBland", "sparsity", "varsize"
    ]

    df = pd.DataFrame(results, columns=columns)
    df[["B0", "B1"]] = pd.DataFrame(df["B_params"].tolist(), index=df.index)
    df["tensor_dim"] = df["size"].apply(lambda x: len(x))
    df = df.drop(columns=["B_params"])

    return df

In [ ]:
df_ = amostragem(100, (10,10), (0.5,0.1), 0)
df_

##### **Desenvolvimento dos Experimentos**



In [ ]:
def experimentos(sample_size, sizeS, beta_paramsS, sparsityS):

    all_results = []  # List to store DataFrames from each experiment

    time_ = time.time()

    for size in sizeS:
        cur = amostragem(sample_size, size=size).values
        new_time_ = time.time()
        print("size=", size, "   ", "duration=", new_time_ - time_)
        time_ = new_time_
        all_results.append(pd.DataFrame(cur))

    # Iterate over beta_paramsS
    for beta_params in beta_paramsS:
        cur = amostragem(sample_size, B_params=beta_params).values
        new_time_ = time.time()
        print("betas=", beta_params, "   ", "duration=", new_time_ - time_)
        time_ = new_time_
        all_results.append(pd.DataFrame(cur))

    # Iterate over sparsityS
    for sparsity in sparsityS:
        cur = amostragem(sample_size, sparsity=sparsity).values
        new_time_ = time.time()
        print("sparsity=", sparsity, "   ", "duration=", new_time_ - time_)
        time_ = new_time_
        all_results.append(pd.DataFrame(cur))

    # Combine all DataFrames into one
    df = pd.concat(all_results, ignore_index=True)

    df = pd.DataFrame(df)

    return df

In [ ]:
#Lista de parâmetros
sizeS = [ (3, 4), (6, 10), (6, 60),
          (2, 3, 4), (2, 5, 6), (3, 3, 40)]
beta_paramsS = [(0.5, 0.001), (0.5, 0.1), (0.5, 0.3), (0.1, 0.05), (0.9, 0.05)]
sparsityS = [0, 0.2, 0.5, 0.9]
#DataFrame com experimentos
df = experimentos(50, sizeS, beta_paramsS, sparsityS)
df.columns = df_.columns
df

In [ ]:
#Passando dados para csv
df.to_csv('dados.csv')

### **Resultados: Gráficos**

In [ ]:
#Criando cópia de df
df_graph = df.copy()
df_graph

In [ ]:
# Convertendo a coluna 'size' para string no DataFrame df_graph
df_graph['size'] = df_graph['size'].astype(str)

##### **1. Comparação SciPy**

In [ ]:
plt.figure(figsize=(12, 6))

# Boxplot para os tempos
plt.subplot(1, 2, 1)  # Primeiro subplot (1 linha, 2 colunas, posição 1)
plt.boxplot([df_["tempo_SimpleXBland"], df_["tempo_scipy"]], labels=["Tempo do SimpleX Bland", "Tempo Scipy"])
plt.title("Comparação dos Tempos de Execução")
plt.ylabel("Tempo (segundos)")
plt.grid(axis="y")

# Boxplot para as iterações
plt.subplot(1, 2, 2)  # Segundo subplot (1 linha, 2 colunas, posição 2)
plt.boxplot([df_["nit_SimpleXBland"], df_["nit_scipy"]], labels=["Iterações SimpleX Bland", "Iterações Scipy"])
plt.title("Comparação do Número de Iterações")
plt.ylabel("Número de Iterações")
plt.grid(axis="y")

plt.tight_layout()
plt.show()


##### **2. Correlação**

In [ ]:
def calculate_complexity(row):
    if row['tensor_dim'] == 2:
        n, m = map(int, row['size'].strip("()").split(","))
        return (n * m)*(1-row['sparsity']) + n + m
    else:  # Assumindo tensor_dim == 3
        n, m, p = map(int, row['size'].strip("()").split(","))
        return (n * m * p)*(1-row['sparsity']) + n + m + p  # Retorna apenas o produto

In [ ]:
# Aplicando a função diretamente
df_graph["complexity"] = df_graph.apply(calculate_complexity, axis=1)

In [ ]:
# Calculando a matriz de correlação
df_graph_no_str = df_graph.drop(['size'], axis=1)
correlation_matrix_graph = df_graph_no_str.corr()

In [ ]:
# Plotando o heatmap da matriz de correlação
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix_graph, annot=True, annot_kws={"size": 8}, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Heatmap da Matriz de Correlação")
plt.show()

##### **3. Variante: Size**

In [ ]:
subset_to_keep = df_graph[(df_graph['size'] == '(6, 10)') & (df_graph['sparsity'] == 0) & (df_graph['B0'] == 0.5) & (df_graph['B1'] == 0.1)]

# Removendo todos os registros com 'size' igual a '(6, 10)'
df_filtered = df_graph[df_graph['size'] != '(6, 10)']

# Adicionando os 50 registros filtrados novamente ao DataFrame
df_result = pd.concat([df_filtered, subset_to_keep], ignore_index=True)
df_result

In [ ]:
# Agrupando os dados por 'complexity' e calculando as médias
grouped_data = df_result.groupby('size')[['tempo_scipy', 'tempo_SimpleXBland']].mean()

# Criando os subplots para os dois gráficos lado a lado
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Sem limite no eixo y
axs[0].plot(grouped_data.index, grouped_data['tempo_scipy'], marker='o', label='Tempo SciPy')
axs[0].plot(grouped_data.index, grouped_data['tempo_SimpleXBland'], marker='o', label='Tempo Simplex Bland')
axs[0].set_title("Aumento do Tempo de Execução com o Tamanho")
axs[0].set_xlabel("Tamanho (complexity)")
axs[0].set_ylabel("Tempo Médio de Execução (segundos)")
axs[0].legend()
axs[0].grid(True)

# Gráfico 2: Com limite no eixo y
axs[1].plot(grouped_data.index, grouped_data['tempo_scipy'], marker='o', label='Tempo SciPy')
axs[1].plot(grouped_data.index, grouped_data['tempo_SimpleXBland'], marker='o', label='Tempo Simplex Bland')
axs[1].set_title("Aumento do Tempo de Execução com o Tamanho (Zoom In: 0 a 0.5)")
axs[1].set_xlabel("Tamanho (complexity)")
axs[1].set_ylabel("Tempo Médio de Execução (segundos)")
axs[1].set_ylim(0, 0.5)
axs[1].legend()
axs[1].grid(True)

# Ajustando o layout
plt.tight_layout()
plt.show()


##### **4. Variante: Sparsity**

In [ ]:
subset_to_keep = df_graph[(df_graph['sparsity'] == 0) & (df_graph['size'] == '(6, 10)') & (df_graph['B0'] == 0.5) & (df_graph['B1'] == 0.1)]

# Removendo todos os registros com 'sparsity' igual a 0
df_filtered = df_graph[df_graph['sparsity'] != 0]

# Adicionando os 50 registros filtrados novamente ao DataFrame
df_result = pd.concat([df_filtered, subset_to_keep], ignore_index=True)
df_result

In [ ]:
# Agrupando os dados por 'complexity' e calculando as médias
grouped_data = df_result.groupby('sparsity')[['tempo_scipy', 'tempo_SimpleXBland']].mean()

# Plotando o gráfico
plt.figure(figsize=(10, 6))
plt.plot(grouped_data.index, grouped_data['tempo_scipy'], marker='o', label='Tempo SciPy')
plt.plot(grouped_data.index, grouped_data['tempo_SimpleXBland'], marker='o', label='Tempo Simplex Bland')

# Configurando o gráfico
plt.title("Aumento do Tempo de Execução com a Espacidade")
plt.xlabel("Espacidade (decimal 0 a 1)")
plt.ylabel("Tempo Médio de Execução (segundos)")
plt.legend()
plt.grid(True)
plt.show()

##### **5.Boxplot: Tempos por Tamanho**

In [ ]:
# Normalizando o tempo por complexidade
df_graph['time_complexity'] = df_graph['tempo_scipy'] / df_graph['complexity'] / (df_graph['tempo_scipy'] / df_graph['complexity']).mean()

In [ ]:
plt.figure(figsize=(10, 6))

# Boxplot para tempo_scipy normalizado
sns.boxplot(x='size', y='time_complexity', data=df_graph)
plt.title("Boxplot do Tempo SciPy por Tamanho")
plt.xlabel("Tamanho (size)")
plt.ylabel("Tempo de Execução (segundos)")
plt.grid(axis='y')

##### **6. ScatterPlot: Tempo de Execução do Simplex x Esparsidade**

In [ ]:
df_result = df_graph[(df_graph['size'] == '(6, 10)') & (df_graph['B0'] == 0.5) & (df_graph['B1'] == 0.1)]

In [ ]:
plt.figure(figsize=(8, 6))

# Calculando médias e desvios padrão para cada nível de esparsidade
grouped = df_result.groupby('sparsity')['tempo_SimpleXBland']
means = grouped.mean()
stds = grouped.std()

# Criando o gráfico de linha com barras de erro
plt.errorbar(x=means.index, y=means, yerr=stds, fmt='-o', capsize=5, label='Tempo Médio com Desvio Padrão')

# Configurando o gráfico
plt.title("Gráfico de Linha: Tempo de Execução do Simplex Bland vs Esparsidade")
plt.xlabel("Esparsidade")
plt.ylabel("Tempo de Execução (segundos)")
plt.legend()
plt.grid(True)
plt.show()